# Phase 5: MIL-Guided Segment Extraction
## Explainable Video Anomaly Detection Pipeline

This notebook implements **Phase 5** of our pipeline: extracting frames from anomalous segments using MIL attention guidance.

---

### Pipeline Position
```
Phase 1: Frame Extraction      ✅ COMPLETE
Phase 2: TimeSformer Features  ✅ COMPLETE  
Phase 3: MIL Training          ✅ COMPLETE (96% Accuracy)
Phase 4: Inference Pipeline    ✅ COMPLETE
Phase 5: Segment Extraction    ◀ YOU ARE HERE
Phase 6: CLIP + InstructBLIP   ⏳ Next
Phase 7: Intelligence Report   ⏳ Next
```

---

### What This Phase Does
1. **Takes Phase 4 output** (`segment_scores`, `anomaly_segments`)
2. **Maps segments to frame ranges** using your parameters (NUM_SEGMENTS=16)
3. **Extracts all frames** from anomalous regions
4. **Selects key frames** for Three-Stage Captioning (Before/Peak/After)
5. **Outputs** frames ready for Phase 6 (Classification & Captioning)

---

### GPU Requirement
**⚡ NO GPU NEEDED** - This phase uses only CPU for video frame extraction (OpenCV operations).

---

### Input (from Phase 4 - DICT format)
```python
# Phase 4 outputs detailed_results.json as DICT (keyed by video_name)
phase4_results = {
    "Fighting001_x264": {
        "video_name": "Fighting001_x264",
        "video_score": 0.94,
        "is_anomaly": True,
        "segment_scores": [0.1, 0.08, ..., 0.89, 0.94, 0.91, 0.85, ...],  # 16 values
        "anomaly_segments": [11, 12, 13, 14],
        "class_name": "Fighting",
        "ground_truth": 1
    },
    ...
}
```

### Output (for Phase 6)
```python
phase5_result = {
    'all_frames': [...],           # All extracted frames
    'key_frames': {                # For Three-Stage Captioning
        'before': frame_array,
        'peak': frame_array,
        'after': frame_array
    },
    'metadata': {...}              # Timing and segment info
}
```

---
## Section 1: Environment Setup & Imports
---

In [ ]:
"""
Phase 5: MIL-Guided Segment Extraction Environment Setup and Imports

NO GPU REQUIRED - This phase uses CPU only for frame extraction.
"""

import os
import sys
import cv2
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional, Tuple, Union
import json
from datetime import datetime
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Version info
print("="*70)
print("PHASE 5: MIL-GUIDED SEGMENT EXTRACTION")
print("="*70)
print(f"\n📦 Package Versions:")
print(f"   Python: {sys.version.split()[0]}")
print(f"   OpenCV: {cv2.__version__}")
print(f"   NumPy: {np.__version__}")
print(f"\n⚡ GPU Required: NO (CPU-based frame extraction)")
print("="*70)

---
## Section 2: Configuration
---

**Important:** These parameters MUST match your Phase 1-4 settings!

In [ ]:
"""
Configuration for Phase 5: MIL-Guided Segment Extraction

CRITICAL: These parameters must match your Phase 1-4 settings!
"""

# ==================== Path Configuration ====================

DATASET_ROOT = r"/kaggle/input/ucf-crime-datasets"
RAW_VIDEOS_PATH = os.path.join(DATASET_ROOT, "Raw_Videos_Unified/Raw_Videos_Unified")
PHASE5_OUTPUT_PATH = "/kaggle/working/Phase5_Extracted_Segments"

# Phase 4 Results Path (from your inference)
PHASE4_JSON_PATH = "/kaggle/input/inference-results/Inference_Results/detailed_results.json"

# ==================== Parameters (MUST MATCH PHASE 1-4) ====================
# From Phase 1
SEQ_LEN = 16          # Frames per clip
STRIDE = 5            # Dilation
CLIP_STEP = 64        # Sliding window jump
IMG_SIZE = 224        # Frame size

# From Phase 3-4
NUM_SEGMENTS = 16     # Fixed segments per video (CRITICAL!)
ANOMALY_THRESHOLD = 0.5  # Threshold for anomaly detection

# ==================== Phase 5 Specific Parameters ====================
# Key frame selection positions (as fraction of anomaly segment)
BEFORE_POSITION = 0.15    # ~15% into the anomaly segment (early context)
PEAK_POSITION = 0.50      # ~50% (middle - main event)
AFTER_POSITION = 0.85     # ~85% (aftermath)

# Frame extraction settings
MAX_FRAMES_TO_STORE = 500      # Maximum frames to store in memory
FRAME_SAMPLE_RATE = 1          # Extract every Nth frame (1 = all frames)
OUTPUT_FRAME_SIZE = (224, 224) # Size for extracted frames

# Batch processing settings
LIMIT_PER_CLASS = 15  # 15 videos x 13 classes = 195 reports

# Create output directory
os.makedirs(PHASE5_OUTPUT_PATH, exist_ok=True)

# ==================== Print Configuration ====================
print("\n" + "="*70)
print("PHASE 5 CONFIGURATION")
print("="*70)
print(f"\n📁 Paths:")
print(f"   Dataset Root: {DATASET_ROOT}")
print(f"   Raw Videos: {RAW_VIDEOS_PATH}")
print(f"   Phase 4 Results: {PHASE4_JSON_PATH}")
print(f"   Output: {PHASE5_OUTPUT_PATH}")
print(f"\n🔢 Parameters (from Phase 1-4):")
print(f"   NUM_SEGMENTS: {NUM_SEGMENTS}")
print(f"   ANOMALY_THRESHOLD: {ANOMALY_THRESHOLD}")
print(f"   IMG_SIZE: {IMG_SIZE}")
print(f"\n🎯 Key Frame Positions:")
print(f"   Before: {BEFORE_POSITION*100:.0f}% into segment")
print(f"   Peak: {PEAK_POSITION*100:.0f}% into segment")
print(f"   After: {AFTER_POSITION*100:.0f}% into segment")
print(f"\n📊 Batch Settings:")
print(f"   Videos per class: {LIMIT_PER_CLASS}")

# Verify Phase 4 JSON exists
if os.path.exists(PHASE4_JSON_PATH):
    print(f"\n✅ Phase 4 JSON found!")
    with open(PHASE4_JSON_PATH, 'r') as f:
        p4_data = json.load(f)
    print(f"   Total videos: {len(p4_data)}")
    anomaly_count = sum(1 for v in p4_data.values() if v.get('is_anomaly'))
    print(f"   Anomalies: {anomaly_count}")
else:
    print(f"\n❌ Phase 4 JSON NOT FOUND: {PHASE4_JSON_PATH}")
print("="*70)

In [ ]:
"""
Verify that all required paths exist
"""

def verify_paths():
    """Check if all required directories exist."""
    print("\n" + "="*70)
    print("PATH VERIFICATION")
    print("="*70)

    paths_to_check = {
        "Dataset Root": DATASET_ROOT,
        "Raw Videos": RAW_VIDEOS_PATH,
    }

    all_exist = True
    for name, path in paths_to_check.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        print(f"   {status} {name}: {path}")
        if not exists:
            all_exist = False

    # Check output directory (create if needed)
    os.makedirs(PHASE5_OUTPUT_PATH, exist_ok=True)
    print(f"   ✅ Output (created): {PHASE5_OUTPUT_PATH}")

    print("\n" + "="*70)
    if all_exist:
        print("✅ ALL PATHS VERIFIED!")
    else:
        print("❌ SOME PATHS MISSING - Please update DATASET_ROOT")
    print("="*70)

    return all_exist

# Run verification
paths_ok = verify_paths()

---
## Section 3: Core Extraction Functions
---

These are the main functions for MIL-guided segment extraction.

In [ ]:
"""
Video Utility Functions
"""

def get_video_info(video_path: str) -> Dict:

    """
    Get video metadata (frame count, fps, duration, resolution).
    Args:
        video_path: Path to video file
    Returns:
        Dictionary with video information
    """
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Video not found: {video_path}")

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    info = {
        'path': video_path,
        'filename': os.path.basename(video_path),
        'total_frames': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
        'fps': cap.get(cv2.CAP_PROP_FPS),
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'duration_seconds': 0
    }

    if info['fps'] > 0:
        info['duration_seconds'] = info['total_frames'] / info['fps']

    cap.release()
    return info


def frame_to_timestamp(frame_idx: int, fps: float) -> str:
    """ Convert frame index to timestamp (HH:MM:SS). """
    if fps <= 0:
        return "00:00:00"

    total_seconds = frame_idx / fps
    hours = int(total_seconds // 3600)
    minutes = int((total_seconds % 3600) // 60)
    seconds = int(total_seconds % 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def extract_single_frame(video_path: str, frame_idx: int,
                         resize: Tuple[int, int] = None) -> Optional[np.ndarray]:
    """ Extract a single frame from video. """
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        return None

    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()

    if not ret or frame is None:
        return None

    # Convert BGR to RGB
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    if resize is not None:
        frame = cv2.resize(frame, resize, interpolation=cv2.INTER_LINEAR)

    return frame


print("✅ Video utility functions defined!")

In [ ]:
"""
Segment-to-Frame Mapping Functions

These functions convert MIL segment indices to actual frame ranges.
"""

def segment_to_frame_range(segment_idx: int, total_frames: int,
                           num_segments: int = NUM_SEGMENTS) -> Tuple[int, int]:
    """
    Convert a segment index to frame range.

    Your MIL model divides videos into NUM_SEGMENTS (16) equal parts.
    This function maps segment index back to frame indices.

    Args:
        segment_idx: Segment index (0 to NUM_SEGMENTS-1)
        total_frames: Total frames in video
        num_segments: Number of segments (default: 16)

    Returns:
        Tuple of (start_frame, end_frame)
    """
    frames_per_segment = total_frames / num_segments

    start_frame = int(segment_idx * frames_per_segment)
    end_frame = int((segment_idx + 1) * frames_per_segment) - 1

    end_frame = min(end_frame, total_frames - 1)

    return start_frame, end_frame


def get_anomaly_frame_ranges(anomaly_segments: List[int], total_frames: int,
                              num_segments: int = NUM_SEGMENTS) -> List[Tuple[int, int]]:
    """
    Convert list of anomaly segment indices to frame ranges.
    """
    frame_ranges = []

    for seg_idx in anomaly_segments:
        start, end = segment_to_frame_range(seg_idx, total_frames, num_segments)
        frame_ranges.append((start, end))

    return frame_ranges


def merge_consecutive_ranges(frame_ranges: List[Tuple[int, int]],
                              gap_threshold: int = 10) -> List[Tuple[int, int]]:
    """
    Merge consecutive or overlapping frame ranges.
    """
    if not frame_ranges:
        return []

    sorted_ranges = sorted(frame_ranges, key=lambda x: x[0])
    merged = [sorted_ranges[0]]

    for current_start, current_end in sorted_ranges[1:]:
        last_start, last_end = merged[-1]

        if current_start <= last_end + gap_threshold:
            merged[-1] = (last_start, max(last_end, current_end))
        else:
            merged.append((current_start, current_end))

    return merged


print("✅ Segment mapping functions defined!")
print(f"\n📝 Example: Video with 4800 frames, {NUM_SEGMENTS} segments")
print(f"   Segment 0: frames {segment_to_frame_range(0, 4800)}")
print(f"   Segment 7: frames {segment_to_frame_range(7, 4800)}")
print(f"   Segment 15: frames {segment_to_frame_range(15, 4800)}")

In [ ]:
"""
MIL-Guided Segment Extractor Class

This is the main class for Phase 5 - extracts frames from anomalous segments
identified by your Phase 4 MIL model.
"""

class MILGuidedSegmentExtractor:
    """
    Extracts frames from video segments identified as anomalous by MIL.
    """

    def __init__(
        self,
        num_segments: int = NUM_SEGMENTS,
        before_position: float = BEFORE_POSITION,
        peak_position: float = PEAK_POSITION,
        after_position: float = AFTER_POSITION,
        frame_size: Tuple[int, int] = OUTPUT_FRAME_SIZE,
        max_frames: int = MAX_FRAMES_TO_STORE,
        sample_rate: int = FRAME_SAMPLE_RATE
    ):
        self.num_segments = num_segments
        self.before_position = before_position
        self.peak_position = peak_position
        self.after_position = after_position
        self.frame_size = frame_size
        self.max_frames = max_frames
        self.sample_rate = sample_rate

    def extract(
        self,
        video_path: str,
        phase4_result: Dict,
        extract_all_frames: bool = True,
        verbose: bool = True
    ) -> Dict:
        """
        Main extraction method.

        Args:
            video_path: Path to the original video file
            phase4_result: Output dictionary from Phase 4
            extract_all_frames: If True, extract all frames
            verbose: Print progress information

        Returns:
            Dictionary with key_frames, metadata, etc.
        """
        # Validate inputs
        if not os.path.exists(video_path):
            raise FileNotFoundError(f"Video not found: {video_path}")

        required_keys = ['anomaly_segments', 'segment_scores', 'video_score', 'is_anomaly']
        for key in required_keys:
            if key not in phase4_result:
                raise ValueError(f"Phase 4 result missing: '{key}'")

        # Get video info
        video_info = get_video_info(video_path)
        total_frames = video_info['total_frames']
        fps = video_info['fps']

        if verbose:
            print(f"\n{'='*60}")
            print(f"MIL-GUIDED SEGMENT EXTRACTION")
            print(f"{'='*60}")
            print(f"\n📹 Video: {video_info['filename']}")
            print(f"   Frames: {total_frames}, FPS: {fps:.2f}")
            print(f"   Duration: {video_info['duration_seconds']:.1f}s")

        # Get Phase 4 results
        is_anomaly = phase4_result['is_anomaly']
        anomaly_segments = phase4_result['anomaly_segments']
        segment_scores = phase4_result['segment_scores']
        video_score = phase4_result['video_score']

        if verbose:
            print(f"\n🎯 Phase 4 Results:")
            print(f"   Score: {video_score:.4f}, Anomaly: {is_anomaly}")
            print(f"   Anomaly Segments: {anomaly_segments}")

        # Handle normal videos
        if not is_anomaly or len(anomaly_segments) == 0:
            if verbose:
                print(f"\n⚠️ No anomaly - extracting middle frames")
            return self._extract_normal_video(video_path, video_info, phase4_result)

        # Map segments to frames
        frame_ranges = get_anomaly_frame_ranges(anomaly_segments, total_frames)
        merged_ranges = merge_consecutive_ranges(frame_ranges)

        if verbose:
            print(f"\n📊 Frame Mapping:")
            for seg, (start, end) in zip(anomaly_segments, frame_ranges):
                print(f"   Segment {seg}: {start}-{end}")

        # Calculate overall range
        overall_start = merged_ranges[0][0]
        overall_end = merged_ranges[-1][1]

        # Select key frame indices
        key_indices = self._calculate_key_frame_indices(
            overall_start, overall_end, segment_scores, anomaly_segments, total_frames
        )

        if verbose:
            print(f"\n🎬 Key Frames:")
            for k, v in key_indices.items():
                print(f"   {k}: Frame {v} ({frame_to_timestamp(v, fps)})")

        # Extract key frames
        key_frames = {}
        for frame_type, frame_idx in key_indices.items():
            frame = extract_single_frame(video_path, frame_idx, self.frame_size)
            if frame is not None:
                key_frames[frame_type] = frame

        # Extract all frames (optional)
        all_frames, all_indices = [], []
        if extract_all_frames:
            all_frames, all_indices = self._extract_frames_from_ranges(
                video_path, merged_ranges, verbose
            )

        # Build result
        timestamps = {k: frame_to_timestamp(v, fps) for k, v in key_indices.items()}

        result = {
            'key_frames': key_frames,
            'all_frames': all_frames,
            'frame_indices': {'key_frames': key_indices, 'all_frames': all_indices},
            'timestamps': timestamps,
            'metadata': {
                'video_path': video_path,
                'video_info': video_info,
                'phase4_result': {
                    'video_score': video_score,
                    'is_anomaly': is_anomaly,
                    'anomaly_segments': anomaly_segments,
                    'segment_scores': list(segment_scores) if hasattr(segment_scores, '__iter__') else segment_scores
                },
                'anomaly_region': {
                    'start_frame': overall_start,
                    'end_frame': overall_end,
                    'start_time': frame_to_timestamp(overall_start, fps),
                    'end_time': frame_to_timestamp(overall_end, fps),
                    'duration_frames': overall_end - overall_start + 1,
                    'duration_seconds': (overall_end - overall_start + 1) / fps if fps > 0 else 0
                },
                'frame_ranges': merged_ranges,
                'num_frames_extracted': len(all_frames),
                'extraction_timestamp': datetime.now().isoformat()
            }
        }

        if verbose:
            print(f"\n{'='*60}")
            print(f"✅ EXTRACTION COMPLETE!")
            print(f"{'='*60}")

        return result

    def _calculate_key_frame_indices(self, start_frame, end_frame, segment_scores,
                                      anomaly_segments, total_frames) -> Dict[str, int]:
        """Calculate indices for before/peak/after frames."""
        duration = end_frame - start_frame

        before_idx = int(start_frame + duration * self.before_position)

        # Peak: middle of highest-scoring segment
        if anomaly_segments and segment_scores:
            scores_arr = np.array(segment_scores)
            seg_scores = [(s, scores_arr[s]) for s in anomaly_segments if s < len(scores_arr)]
            if seg_scores:
                peak_seg = max(seg_scores, key=lambda x: x[1])[0]
                ps, pe = segment_to_frame_range(peak_seg, total_frames)
                peak_idx = (ps + pe) // 2
            else:
                peak_idx = int(start_frame + duration * self.peak_position)
        else:
            peak_idx = int(start_frame + duration * self.peak_position)

        after_idx = int(start_frame + duration * self.after_position)

        # Bounds check
        before_idx = max(0, min(before_idx, total_frames - 1))
        peak_idx = max(0, min(peak_idx, total_frames - 1))
        after_idx = max(0, min(after_idx, total_frames - 1))

        # Ensure ordering
        if before_idx >= peak_idx:
            before_idx = max(0, peak_idx - 10)
        if after_idx <= peak_idx:
            after_idx = min(total_frames - 1, peak_idx + 10)

        return {'before': before_idx, 'peak': peak_idx, 'after': after_idx}

    def _extract_frames_from_ranges(self, video_path, frame_ranges, verbose):
        """Extract frames from specified ranges."""
        frames, indices = [], []
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return frames, indices

        total = sum(e - s + 1 for s, e in frame_ranges)
        sample_rate = self.sample_rate
        if total > self.max_frames * sample_rate:
            sample_rate = max(1, total // self.max_frames)

        for start, end in frame_ranges:
            for idx in range(start, end + 1, sample_rate):
                if len(frames) >= self.max_frames:
                    break
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if ret and frame is not None:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    if self.frame_size:
                        frame = cv2.resize(frame, self.frame_size)
                    frames.append(frame)
                    indices.append(idx)

        cap.release()
        return frames, indices

    def _extract_normal_video(self, video_path, video_info, phase4_result):
        """Handle normal videos."""
        total = video_info['total_frames']
        fps = video_info['fps']
        mid = total // 2

        key_indices = {
            'before': max(0, mid - int(fps * 5)),
            'peak': mid,
            'after': min(total - 1, mid + int(fps * 5))
        }

        key_frames = {}
        for ft, fi in key_indices.items():
            frame = extract_single_frame(video_path, fi, self.frame_size)
            if frame is not None:
                key_frames[ft] = frame

        return {
            'key_frames': key_frames,
            'all_frames': [],
            'frame_indices': {'key_frames': key_indices, 'all_frames': []},
            'timestamps': {k: frame_to_timestamp(v, fps) for k, v in key_indices.items()},
            'metadata': {
                'video_path': video_path,
                'video_info': video_info,
                'phase4_result': phase4_result,
                'anomaly_region': None,
                'note': 'Normal video',
                'extraction_timestamp': datetime.now().isoformat()
            }
        }


print("✅ MILGuidedSegmentExtractor class defined!")

---
## Section 4: Visualization Functions
---

In [ ]:
"""
Visualization Functions
"""

def visualize_key_frames(phase5_result: Dict, figsize=(15, 5)):
    """Visualize the three key frames."""
    key_frames = phase5_result['key_frames']
    timestamps = phase5_result['timestamps']

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    types = ['before', 'peak', 'after']
    titles = ['BEFORE (Context)', 'PEAK (Main Event)', 'AFTER (Aftermath)']
    colors = ['#2196F3', '#F44336', '#4CAF50']

    for ax, ft, title, color in zip(axes, types, titles, colors):
        if ft in key_frames:
            ax.imshow(key_frames[ft])
            ax.set_title(f"{title}\n{timestamps.get(ft, '')}", color=color, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center')
            ax.set_title(title)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


def visualize_segment_scores(phase5_result: Dict, figsize=(12, 4)):
    """Visualize segment scores."""
    p4 = phase5_result['metadata'].get('phase4_result', {})
    scores = p4.get('segment_scores', [])
    anomaly_segs = p4.get('anomaly_segments', [])

    if not scores:
        print("No segment scores available.")
        return

    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(scores))
    colors = ['#F44336' if i in anomaly_segs else '#2196F3' for i in x]

    ax.bar(x, scores, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=ANOMALY_THRESHOLD, color='red', linestyle='--', label=f'Threshold ({ANOMALY_THRESHOLD})')

    ax.set_xlabel('Segment')
    ax.set_ylabel('Score')
    ax.set_title(f"Segment Scores (Video: {p4.get('video_score', 0):.3f})")
    ax.set_xticks(x)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()


def visualize_extraction_summary(phase5_result: Dict):
    """Print extraction summary."""
    meta = phase5_result['metadata']
    vi = meta.get('video_info', {})
    p4 = meta.get('phase4_result', {})
    ar = meta.get('anomaly_region', {})
    ts = phase5_result['timestamps']
    ki = phase5_result['frame_indices'].get('key_frames', {})

    print("\n" + "═"*60)
    print("📊 PHASE 5 EXTRACTION SUMMARY")
    print("═"*60)
    print(f"\n📹 Video: {os.path.basename(vi.get('path', 'N/A'))}")
    print(f"   Frames: {vi.get('total_frames', 'N/A')}, Duration: {vi.get('duration_seconds', 0):.1f}s")
    print(f"\n🎯 Detection: Score={p4.get('video_score', 0):.4f}, Anomaly={p4.get('is_anomaly')}")
    print(f"   Segments: {p4.get('anomaly_segments', [])}")
    if ar:
        print(f"\n📍 Region: {ar.get('start_time')} - {ar.get('end_time')} ({ar.get('duration_seconds', 0):.1f}s)")
    print(f"\n🎬 Key Frames:")
    for k in ['before', 'peak', 'after']:
        print(f"   {k.capitalize()}: Frame {ki.get(k, 'N/A')} @ {ts.get(k, 'N/A')}")
    print("═"*60)


print("✅ Visualization functions defined!")

---
## Section 5: Save/Load Functions
---

In [ ]:
"""
Save/Load Functions
"""

def save_phase5_result(phase5_result: Dict, output_dir: str, video_name: str) -> str:
    """Save Phase 5 results to disk."""
    video_dir = os.path.join(output_dir, video_name)
    os.makedirs(video_dir, exist_ok=True)

    # Save key frames
    for ft, frame in phase5_result.get('key_frames', {}).items():
        if frame is not None:
            path = os.path.join(video_dir, f"{ft}.jpg")
            cv2.imwrite(path, cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    # Save metadata
    meta = {
        'frame_indices': phase5_result.get('frame_indices', {}),
        'timestamps': phase5_result.get('timestamps', {}),
        'metadata': phase5_result.get('metadata', {})
    }
    with open(os.path.join(video_dir, 'metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    # Save all frames if present
    all_frames = phase5_result.get('all_frames', [])
    if 0 < len(all_frames) <= MAX_FRAMES_TO_STORE:
        np.save(os.path.join(video_dir, 'all_frames.npy'), np.array(all_frames))

    print(f"✅ Saved to: {video_dir}")
    return video_dir


def load_phase5_result(result_dir: str) -> Dict:
    """Load Phase 5 results from disk."""
    with open(os.path.join(result_dir, 'metadata.json'), 'r') as f:
        data = json.load(f)

    key_frames = {}
    for ft in ['before', 'peak', 'after']:
        path = os.path.join(result_dir, f"{ft}.jpg")
        if os.path.exists(path):
            key_frames[ft] = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

    all_frames = []
    frames_path = os.path.join(result_dir, 'all_frames.npy')
    if os.path.exists(frames_path):
        all_frames = list(np.load(frames_path))

    return {
        'key_frames': key_frames,
        'all_frames': all_frames,
        **data
    }


print("✅ Save/Load functions defined!")

---
## Section 6: Integrated Batch Execution
---

This section processes **10 videos per class** directly from your Phase 4 results, creating the 130-video dataset for Pre-Thesis 2.

In [ ]:
"""
Integrated Batch Phase 5 Execution
Processes 10 videos per class from Phase 4 results (130 total reports)

NOTE: Phase 4 outputs DICT format (keyed by video_name)
"""

from collections import defaultdict

def run_integrated_phase5_batch():
    """
    Run Phase 5 extraction on all anomaly videos from Phase 4.
    Stratified sampling: 10 videos per class.

    Phase 4 JSON Format (DICT):
    {
        "video_name": {
            "video_name": "...",
            "video_score": 0.94,
            "is_anomaly": True,
            "class_name": "Fighting",
            ...
        }
    }
    """
    class_counts = defaultdict(int)
    total_processed = 0
    skipped_normal = 0
    skipped_limit = 0
    not_found = 0
    errors = 0

    extractor = MILGuidedSegmentExtractor()

    # Verify Phase 4 results exist
    if not os.path.exists(PHASE4_JSON_PATH):
        print(f"❌ Error: Phase 4 JSON not found at {PHASE4_JSON_PATH}")
        print("   Please ensure you have the inference results from Phase 4.")
        return None

    # Load Phase 4 results (DICT format)
    print(f"\n📂 Loading Phase 4 results from: {PHASE4_JSON_PATH}")
    with open(PHASE4_JSON_PATH, 'r') as f:
        all_results = json.load(f)

    print(f"   Found {len(all_results)} video results")
    print(f"   Format: DICT (keyed by video_name)")

    # Count anomalies per class
    class_anomalies = defaultdict(int)
    for video_name, data in all_results.items():
        if data.get('is_anomaly'):
            cls = data.get('class_name', 'Unknown')
            class_anomalies[cls] += 1

    print(f"\n📊 Anomalies available per class:")
    for cls in sorted(class_anomalies.keys()):
        print(f"   {cls}: {class_anomalies[cls]}")

    print(f"\n🚀 Starting Batch Phase 5: Processing {LIMIT_PER_CLASS} videos per class...")
    print("="*60)

    # Iterate over DICT items
    for video_name, data in tqdm(all_results.items(), desc="Processing Videos"):
        # Process only videos flagged as anomalies by Phase 4
        if not data.get('is_anomaly'):
            skipped_normal += 1
            continue

        # Get class from Phase 4 result (preferred) or extract from filename
        video_class = data.get('class_name')
        if not video_class:
            # Fallback: Extract class from filename (e.g., 'Fighting052_x264' -> 'Fighting')
            base_name = video_name.replace('.mp4', '').replace('.avi', '')
            video_class = "".join([c for c in base_name.split('_')[0] if not c.isdigit()])

        # Skip if we've reached the limit for this class
        if class_counts[video_class] >= LIMIT_PER_CLASS:
            skipped_limit += 1
            continue

        # Get the base name for file matching
        base_name = video_name.replace('.mp4', '').replace('.avi', '')

        # Search for the video file in dataset subfolders
        video_path = None
        for root, _, files in os.walk(RAW_VIDEOS_PATH):
            for f in files:
                # Match with or without extension
                f_base = f.replace('.mp4', '').replace('.avi', '')
                if f_base == base_name or f == video_name:
                    video_path = os.path.join(root, f)
                    break
            if video_path:
                break

        if not video_path:
            not_found += 1
            if not_found <= 3:  # Only print first 3 not found
                print(f"   ⚠️ Video not found: {video_name}")
            continue

        try:
            # Run MIL-Guided extraction
            result = extractor.extract(
                video_path=video_path,
                phase4_result=data,
                extract_all_frames=True,
                verbose=False
            )

            # Save frames and metadata for Phase 6
            output_name = base_name
            save_phase5_result(result, PHASE5_OUTPUT_PATH, output_name)

            class_counts[video_class] += 1
            total_processed += 1

            # Visual verification for the first video processed
            if total_processed == 1:
                print(f"\n🔍 First video verification: {video_name}")
                print(f"   Class: {video_class}")
                print(f"   Score: {data.get('video_score', 0):.4f}")
                print(f"   Anomaly Segments: {data.get('anomaly_segments', [])}")
                visualize_key_frames(result)
                visualize_segment_scores(result)

        except Exception as e:
            errors += 1
            if errors <= 5:  # Only print first 5 errors
                print(f"⚠️ Error with {video_name}: {str(e)[:100]}")

    # Print summary
    print("\n" + "="*60)
    print("📊 PHASE 5 BATCH PROCESSING SUMMARY")
    print("="*60)
    print(f"\n✅ Successfully processed: {total_processed} videos")
    print(f"⏭️  Skipped (normal videos): {skipped_normal}")
    print(f"⏭️  Skipped (class limit reached): {skipped_limit}")
    print(f"❓ Not found: {not_found}")
    print(f"❌ Errors: {errors}")

    print(f"\n📁 Videos per class:")
    for cls in sorted(class_counts.keys()):
        count = class_counts[cls]
        bar = "█" * count + "░" * (LIMIT_PER_CLASS - count)
        print(f"   {cls:15s}: [{bar}] {count}/{LIMIT_PER_CLASS}")

    print(f"\n📂 Output saved to: {PHASE5_OUTPUT_PATH}")
    print("="*60)

    return {
        'total_processed': total_processed,
        'class_counts': dict(class_counts),
        'skipped_normal': skipped_normal,
        'skipped_limit': skipped_limit,
        'not_found': not_found,
        'errors': errors
    }


print("✅ Integrated batch function defined!")
print(f"\n📝 Ready to process {LIMIT_PER_CLASS} videos per class")
print(f"   Expected output: ~130 video extractions (10 × 13 classes)")
print(f"   Phase 4 format: DICT (keyed by video_name)")

In [ ]:
"""
Execute Integrated Batch Phase 5
This cell runs the full extraction pipeline on your Phase 4 results.
"""

# Run the batch extraction
batch_results = run_integrated_phase5_batch()

In [ ]:
"""
Verify Output Structure
Check that the extracted frames are correctly organized for Phase 6.
"""

def verify_phase5_output():
    """Verify the output structure is ready for Phase 6."""
    print("\n" + "="*60)
    print("📋 PHASE 5 OUTPUT VERIFICATION")
    print("="*60)

    if not os.path.exists(PHASE5_OUTPUT_PATH):
        print(f"❌ Output directory not found: {PHASE5_OUTPUT_PATH}")
        return

    video_dirs = [d for d in os.listdir(PHASE5_OUTPUT_PATH)
                  if os.path.isdir(os.path.join(PHASE5_OUTPUT_PATH, d))]

    print(f"\n📁 Total video extractions: {len(video_dirs)}")

    # Check structure of first few videos
    complete = 0
    incomplete = 0

    for video_dir in video_dirs[:5]:
        dir_path = os.path.join(PHASE5_OUTPUT_PATH, video_dir)
        files = os.listdir(dir_path)

        has_before = 'before.jpg' in files
        has_peak = 'peak.jpg' in files
        has_after = 'after.jpg' in files
        has_meta = 'metadata.json' in files

        is_complete = all([has_before, has_peak, has_after, has_meta])

        if is_complete:
            complete += 1
            status = "✅"
        else:
            incomplete += 1
            status = "⚠️"

        print(f"\n{status} {video_dir}/")
        print(f"   before.jpg: {'✓' if has_before else '✗'}")
        print(f"   peak.jpg: {'✓' if has_peak else '✗'}")
        print(f"   after.jpg: {'✓' if has_after else '✗'}")
        print(f"   metadata.json: {'✓' if has_meta else '✗'}")

    if len(video_dirs) > 5:
        print(f"\n   ... and {len(video_dirs) - 5} more videos")

    # Count by class
    print("\n📊 Extraction by Class:")
    class_counts = defaultdict(int)
    for vd in video_dirs:
        cls = "".join([c for c in vd.split('_')[0] if not c.isdigit()])
        class_counts[cls] += 1

    for cls in sorted(class_counts.keys()):
        print(f"   {cls}: {class_counts[cls]}")

    print("\n" + "="*60)
    print("✅ Output structure verified - Ready for Phase 6!")
    print("="*60)

# Run verification
verify_phase5_output()

In [ ]:
"""
Sample Output Preview
Visualize a random sample from the extracted videos.
"""

import random

def preview_random_extraction():
    """Preview a random extraction result."""
    if not os.path.exists(PHASE5_OUTPUT_PATH):
        print("❌ No output to preview")
        return

    video_dirs = [d for d in os.listdir(PHASE5_OUTPUT_PATH)
                  if os.path.isdir(os.path.join(PHASE5_OUTPUT_PATH, d))]

    if not video_dirs:
        print("❌ No extractions found")
        return

    # Pick a random video
    sample_video = random.choice(video_dirs)
    sample_path = os.path.join(PHASE5_OUTPUT_PATH, sample_video)

    print(f"\n🎲 Random Sample: {sample_video}")
    print("="*50)

    # Load and display
    result = load_phase5_result(sample_path)

    visualize_key_frames(result)
    visualize_extraction_summary(result)

# Preview a random sample
preview_random_extraction()

---
## Section 7: Phase 6 Integration Helper
---

Utility function to load Phase 5 results for use in Phase 6 (CLIP + InstructBLIP).

In [ ]:
"""
Phase 6 Integration Helper
Provides utilities to load Phase 5 outputs for captioning pipeline.
"""

def get_all_phase5_results(output_path: str = PHASE5_OUTPUT_PATH) -> List[Dict]:
    """
    Load all Phase 5 results for Phase 6 processing.

    Returns:
        List of dictionaries containing:
        - video_name: Name of the video
        - key_frames: Dict with before/peak/after frames
        - metadata: Phase 4 scores and timing info
    """
    results = []

    if not os.path.exists(output_path):
        print(f"❌ Phase 5 output not found: {output_path}")
        return results

    video_dirs = sorted([d for d in os.listdir(output_path)
                        if os.path.isdir(os.path.join(output_path, d))])

    for video_dir in tqdm(video_dirs, desc="Loading Phase 5 results"):
        dir_path = os.path.join(output_path, video_dir)

        try:
            result = load_phase5_result(dir_path)
            result['video_name'] = video_dir
            results.append(result)
        except Exception as e:
            print(f"⚠️ Error loading {video_dir}: {e}")

    print(f"\n✅ Loaded {len(results)} video extractions for Phase 6")
    return results


def prepare_phase6_batch(results: List[Dict]) -> List[Dict]:
    """
    Prepare Phase 5 results for Phase 6 batch processing.

    Returns a list of items ready for CLIP classification and InstructBLIP captioning.
    """
    batch = []

    for r in results:
        video_name = r.get('video_name', 'unknown')
        key_frames = r.get('key_frames', {})
        metadata = r.get('metadata', {})
        timestamps = r.get('timestamps', {})

        # Extract class from video name
        video_class = "".join([c for c in video_name.split('_')[0] if not c.isdigit()])

        batch.append({
            'video_name': video_name,
            'video_class': video_class,
            'frames': {
                'before': key_frames.get('before'),
                'peak': key_frames.get('peak'),
                'after': key_frames.get('after')
            },
            'timestamps': timestamps,
            'phase4_score': metadata.get('phase4_result', {}).get('video_score', 0),
            'anomaly_segments': metadata.get('phase4_result', {}).get('anomaly_segments', []),
            'anomaly_region': metadata.get('anomaly_region', {})
        })

    return batch


print("✅ Phase 6 integration helpers defined!")
print("\n📝 Usage in Phase 6:")
print("""
# Load all Phase 5 extractions
phase5_results = get_all_phase5_results()

# Prepare for batch processing
batch = prepare_phase6_batch(phase5_results)

# Process each video
for item in batch:
    before_frame = item['frames']['before']
    peak_frame = item['frames']['peak']
    after_frame = item['frames']['after']

    # Run CLIP classification
    # Run InstructBLIP captioning
    # Generate intelligence report
""")

---
## Section 8: Summary
---

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║           PHASE 5 IMPLEMENTATION COMPLETE                    ║
║              Pre-Thesis 2 Batch Mode                         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  ✅ MIL-Guided Segment Extraction (Batch Mode)               ║
║                                                              ║
║  INPUTS:                                                     ║
║  ├─ Phase 4 JSON: detailed_results.json                      ║
║  ├─ Raw Videos: UCF Crime Dataset                            ║
║  └─ Limit: 10 videos per class (130 total)                   ║
║                                                              ║
║  OUTPUTS (per video):                                        ║
║  ├─ before.jpg  (Early context frame)                        ║
║  ├─ peak.jpg    (Main incident frame)                        ║
║  ├─ after.jpg   (Aftermath frame)                            ║
║  └─ metadata.json (Scores, timestamps, regions)              ║
║                                                              ║
║  OUTPUT STRUCTURE:                                           ║
║  /kaggle/working/Phase5_Extracted_Segments/                  ║
║  ├── Fighting052/                                            ║
║  │   ├── before.jpg                                          ║
║  │   ├── peak.jpg                                            ║
║  │   ├── after.jpg                                           ║
║  │   └── metadata.json                                       ║
║  ├── Explosion001/                                           ║
║  │   └── ...                                                 ║
║  └── ... (130 video folders)                                 ║
║                                                              ║
║  FEATURES:                                                   ║
║  ├─ ✅ No GPU required (CPU-based)                           ║
║  ├─ ✅ Integrated with Phase 4 JSON                          ║
║  ├─ ✅ Stratified sampling (10/class)                        ║
║  ├─ ✅ Three-Stage Temporal Frames                           ║
║  └─ ✅ Ready for Phase 6 batch processing                    ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  NEXT: Phase 6 - Hierarchical CLIP + InstructBLIP            ║
║        Will read from Phase5_Extracted_Segments/             ║
╚══════════════════════════════════════════════════════════════╝
""")